# One-dimensional particle in a box

**Prerequisite:** [One-dimensional particle-in-a-box spectrum](../../../docs/lecture-notes/qm/piab1d/piab1d.md)

**Presentation lineage:** computational particle-in-a-box lecture for SOLST01, AY 2025--2026 T3, by Eugene Joseph M. Ragasa.

This maintained laboratory uses a familiar quantum system to expose the computational representation chain:

```mermaid
flowchart LR
    A["Hamiltonian operator"] --> B["Hamiltonian matrix"]
    C["Continuous wavefunction"] --> D["Grid vector"]
    E["Differential eigenproblem"] --> F["Matrix eigenproblem"]
```

The continuum equation and represented equation are

$$
\hat H|\psi_nangle=E_n|\psi_nangle,
\qquad
Holdsymbol\psi_n=E_noldsymbol\psi_n.
$$

The selected model is nondimensional: $L=1$, $m=1$, and $\hbar=1$. The plots retain the original lecture progression from grid points, through represented spectra and errors, to a continuously normalized eigenfunction and probability density.

## Learning objectives

1. Distinguish fixed Dirichlet boundary values from interior unknowns.
2. Evaluate the analytical stationary-state energies.
3. Solve the finite-dimensional Dirichlet Hamiltonian.
4. Separate the exact spectrum of the represented matrix from its continuum-discretization error.
5. Compare analytical and finite-difference ground-state wavefunctions and probability densities.
6. Verify the continuum eigenvalue equation and normalization symbolically.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from projectkoios.physkit.qm.piab1d.base import Piab1D
from projectkoios.physkit.qm.piab1d.tdse_analytical.solver import (
    Piab1dTdseAnalyticalSolver,
)
from projectkoios.physkit.qm.piab1d.tise_analytical.solution import (
    Piab1DAnalyticalSolution,
)
from projectkoios.physkit.qm.piab1d.tise_fd.solver import Piab1dTiseFdSolver
from projectkoios.physkit.units.quantities import (
    ComplexVectorQuantity,
    Unitless,
    VectorQuantity,
)
from projectkoios.physkit.units.systems import UnitSystem

In [ ]:
model = Piab1D(
    length=1.0,
    mass=1.0,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
mode_count = 10
interior_points = 200

analytical = Piab1DAnalyticalSolution(model=model).evaluate(count=mode_count)
finite_difference = Piab1dTiseFdSolver().solve(
    model=model, interior_points=interior_points
)

analytical_energies = analytical.energies.magnitude
finite_difference_energies = finite_difference.energies.magnitude[:mode_count]
relative_energy_errors = np.abs(
    (finite_difference_energies - analytical_energies) / analytical_energies
)

energy_comparison = np.column_stack(
    (
        analytical.quantum_numbers,
        analytical_energies,
        finite_difference_energies,
        relative_energy_errors,
    )
)
energy_comparison

## Grid representation

The finite-difference interval includes both boundary points, but homogeneous Dirichlet values are fixed at zero. Only the interior samples are unknown components of the represented state vector. The three aligned panels preserve this distinction before any matrix is diagonalized.


In [ ]:
grid_coordinates = finite_difference.interval.grid.coordinates().magnitude
interior_coordinates = finite_difference.interval.grid.interior_coordinates().magnitude
boundary_coordinates = grid_coordinates[[0, -1]]

grid_figure, grid_axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(8.0, 4.5),
    sharex=True,
)
grid_axes[0].scatter(
    grid_coordinates,
    np.zeros_like(grid_coordinates),
    s=12,
    label="all grid points",
)
grid_axes[1].scatter(
    interior_coordinates,
    np.zeros_like(interior_coordinates),
    s=12,
    label="interior unknowns",
)
grid_axes[2].scatter(
    boundary_coordinates,
    np.zeros_like(boundary_coordinates),
    marker="x",
    s=80,
    label="fixed zero boundaries",
)
for axis in grid_axes:
    axis.set_yticks([])
    axis.grid(True)
    axis.legend(loc="upper right")
grid_axes[-1].set_xlabel("reduced coordinate $x/L$")
grid_figure.suptitle("Dirichlet grid representation")
grid_figure.tight_layout()
plt.show()

In [ ]:
x_symbol = sp.Symbol("x", real=True)
length_symbol = sp.Symbol("L", positive=True, real=True)
mass_symbol = sp.Symbol("m", positive=True, real=True)
hbar_symbol = sp.Symbol("hbar", positive=True, real=True)
quantum_number_symbol = sp.Symbol("n", positive=True, integer=True)

symbolic_wavefunction = sp.sqrt(2 / length_symbol) * sp.sin(
    quantum_number_symbol * sp.pi * x_symbol / length_symbol
)
symbolic_energy = (
    hbar_symbol**2
    * sp.pi**2
    * quantum_number_symbol**2
    / (2 * mass_symbol * length_symbol**2)
)
symbolic_hamiltonian_wavefunction = (
    -(hbar_symbol**2) / (2 * mass_symbol) * sp.diff(symbolic_wavefunction, x_symbol, 2)
)
symbolic_residual = sp.simplify(
    symbolic_hamiltonian_wavefunction - symbolic_energy * symbolic_wavefunction
)
symbolic_normalization = sp.simplify(
    sp.integrate(
        symbolic_wavefunction**2,
        (x_symbol, 0, length_symbol),
    )
)

assert symbolic_residual == 0
assert symbolic_normalization == 1
symbolic_energy

In [ ]:
represented_matrix_energies = (
    finite_difference.closed_form_discrete_energies().magnitude[:mode_count]
)

assert np.allclose(
    finite_difference_energies,
    represented_matrix_energies,
    rtol=3e-12,
    atol=2e-13,
)
assert np.all(relative_energy_errors < 2.1e-3)

relative_energy_errors

The eigensolver agrees with the exact eigenvalues of the represented tridiagonal matrix. The remaining difference from the continuum energies is the finite-difference discretization error for this grid.


In [ ]:
initial_coefficients = np.zeros(mode_count, dtype=np.complex128)
initial_coefficients[0] = 1.0 + 0.0j
analytical_ground_state = Piab1dTdseAnalyticalSolver().solve(
    tise_results=analytical,
    initial_coefficients=ComplexVectorQuantity(
        magnitude=initial_coefficients, unit=Unitless()
    ),
    times=VectorQuantity(magnitude=np.array([0.0], dtype=np.float64), unit=Unitless()),
)

coordinates = finite_difference.interval.grid.interior_coordinates()
ground_state_evolution = analytical_ground_state.state_evolution(
    coordinates=coordinates
)
continuum_ground = next(iter(ground_state_evolution)).amplitudes.magnitude.real
spacing = finite_difference.grid_spacing.magnitude
continuum_ground_vector = continuum_ground * np.sqrt(spacing)
finite_difference_ground_vector = finite_difference.eigenvectors.magnitude[:, 0].copy()

if (
    np.vdot(
        continuum_ground_vector,
        finite_difference_ground_vector,
    ).real
    < 0.0
):
    finite_difference_ground_vector *= -1.0

wavefunction_maximum_error = float(
    np.max(np.abs(finite_difference_ground_vector - continuum_ground_vector))
)
assert np.allclose(
    finite_difference_ground_vector,
    continuum_ground_vector,
    rtol=0.0,
    atol=2e-13,
)
wavefunction_maximum_error

In [ ]:
figure, axes_grid = plt.subplots(2, 3, figsize=(16.0, 8.5))
axes = axes_grid.ravel()

axes[0].plot(
    analytical.quantum_numbers,
    analytical_energies,
    marker="o",
    label="continuum analytical",
)
axes[0].scatter(
    analytical.quantum_numbers,
    finite_difference_energies,
    marker="x",
    label="finite difference",
)
axes[0].set(
    xlabel="quantum number $n$",
    ylabel="energy",
    title="Stationary-state energies",
)
axes[0].grid(True)
axes[0].legend()

axes[1].plot(
    analytical.quantum_numbers,
    analytical_energies / analytical_energies[0],
    marker="o",
    label="continuum $E_n/E_1=n^2$",
)
axes[1].scatter(
    analytical.quantum_numbers,
    finite_difference_energies / finite_difference_energies[0],
    marker="x",
    label="finite difference",
)
axes[1].set(
    xlabel="quantum number $n$",
    ylabel="$E_n/E_1$",
    title="Scaled energy spectrum",
)
axes[1].grid(True)
axes[1].legend()

axes[2].semilogy(
    analytical.quantum_numbers,
    relative_energy_errors,
    marker="o",
)
axes[2].set(
    xlabel="quantum number $n$",
    ylabel="relative energy error",
    title="Continuum discretization error",
)
axes[2].grid(True)

axes[3].plot(
    coordinates.magnitude,
    continuum_ground,
    label="continuum analytical",
)
axes[3].scatter(
    coordinates.magnitude[::8],
    (finite_difference_ground_vector / np.sqrt(spacing))[::8],
    marker="x",
    label="finite difference",
)
axes[3].set(
    xlabel="$x/L$",
    ylabel=r"$\psi_1(x)$",
    title="Continuously normalized ground state",
)
axes[3].grid(True)
axes[3].legend()

axes[4].plot(
    coordinates.magnitude,
    np.abs(continuum_ground) ** 2,
)
axes[4].set(
    xlabel="$x/L$",
    ylabel=r"$|\psi_1(x)|^2$",
    title="Ground-state probability density",
)
axes[4].grid(True)

hamiltonian_block_size = 24
hamiltonian_block = np.abs(
    finite_difference.hamiltonian.to_csr().toarray()[
        :hamiltonian_block_size,
        :hamiltonian_block_size,
    ]
)
matrix_image = axes[5].imshow(
    hamiltonian_block,
    origin="lower",
    cmap="magma",
    aspect="equal",
)
axes[5].set(
    xlabel="column index",
    ylabel="row index",
    title="Leading Hamiltonian matrix block",
)
figure.colorbar(matrix_image, ax=axes[5], label="absolute matrix element")

figure.tight_layout()
plt.show()

## Interpretation

For 200 interior points, the first ten finite-difference energies approach the continuum values from below. The scaled spectrum preserves the analytical $n^2$ pattern while the logarithmic error panel makes the increasing finite-grid error visible. The discrete ground-state eigenvector agrees with the analytical sine mode after accounting for vector normalization and arbitrary eigenvector sign; its squared magnitude is the corresponding probability density. The matrix panel exposes the tridiagonal represented Hamiltonian rather than hiding the operator-to-matrix step.

The grid, spectrum, error, wavefunction, probability-density, and matrix panels are pedagogical views of one calculation. Their agreement is numerical verification of this representation, not evidence for a material system.

## Exercises

1. Repeat the calculation with 50, 100, 200, and 400 interior points. Estimate the energy-error convergence rate.
2. Increase `mode_count` and identify where the fixed grid stops resolving the continuum spectrum accurately.
3. Compare the first excited-state eigenvector after applying the same normalization and sign convention.
4. Change the box length and verify the $E_n\propto L^{-2}$ scaling.
